In [43]:
import torch
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset

from PIL import Image
import numpy as np
from torchinfo import summary

from torch import nn, optim
from torch.optim import lr_scheduler

import pandas as pd
import os
import cv2


from tqdm import tqdm 
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


cuda
NVIDIA GeForce RTX 5060


In [44]:
torch.__version__

'2.11.0+cu128'

In [45]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.11.0+cu128
CUDA available: True
CUDA version: 12.8
GPU: NVIDIA GeForce RTX 5060


In [46]:
current_dir = os.getcwd()  # или os.path.dirname(__file__) для .py файлов

# Формируем правильные пути
test_img_dir = os.path.join(current_dir, 'dataset', 'test_images')
train_img_dir = os.path.join(current_dir, 'dataset', 'train_images')
csv_path = os.path.join(current_dir, 'dataset', 'train_solution.csv')

In [47]:
import cv2


def denoise_salt_pepper_on_the_fly(image_np, ksize_median=5, h=8, hColor=8):
    median = cv2.medianBlur(image_np, ksize_median)
    denoised = cv2.fastNlMeansDenoisingColored(
        median,
        None,
        h=h,
        hColor=hColor,
        templateWindowSize=7,
        searchWindowSize=21
    )
    return denoised

In [48]:
class FaceDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.data_frame = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.data_frame)

    def __getitem__(self, idx):
        img_id = str(self.data_frame.iloc[idx, 0]) + '.jpg'
        img_path = os.path.join(self.img_dir, img_id)
        image = Image.open(img_path).convert('RGB')
        label = int(self.data_frame.iloc[idx, 1])
        if self.transform:
            image = self.transform(image)
        return image, label
    

In [49]:
class FaceTestDataset(Dataset):
    def __init__(self, img_dir, transform=None):
        self.img_dir = img_dir
        self.transform = transform
        self.img_names = sorted([f for f in os.listdir(img_dir) if f.endswith('.jpg')])

    def __len__(self):
        return len(self.img_names)

    def __getitem__(self, idx):
        img_name = self.img_names[idx]
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        img_id = img_name.replace('.jpg', '')
        return image, img_id

In [50]:
IMG_SIZE = 256
transform_train = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0))
    ], p=0.3),
    
    transforms.RandomAdjustSharpness(sharpness_factor=2, p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.1)),
])

face_dataset_train = FaceDataset(
    csv_file=csv_path,
    img_dir=train_img_dir,
    transform=transform_train
)

face_dataset_loader = DataLoader(face_dataset_train, batch_size=64, shuffle=True)

In [51]:
transform_valid = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

face_dataset_validation = FaceTestDataset(
    img_dir=test_img_dir,
    transform=transform_valid
)

face_dataset_validation_loader = DataLoader(
    face_dataset_validation,
    batch_size=1,
    shuffle=False
)

In [52]:
class SkipConnectionBlock(nn.Module):
    def __init__(self, in_channels):
        super(SkipConnectionBlock, self).__init__()
        self.model = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(in_channels),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(in_channels)
        )
    
    def forward(self, x):
        return torch.relu(self.model(x) + x)

In [53]:
class MultiScaleBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(MultiScaleBlock, self).__init__()
        
        branch3x3_ch = out_channels // 2
        branch1x1_ch = out_channels // 4
        branch5x5_ch = out_channels - branch3x3_ch - branch1x1_ch
        
        self.branch1x1 = nn.Sequential(
            nn.Conv2d(in_channels, branch1x1_ch, kernel_size=1),
            nn.BatchNorm2d(branch1x1_ch),
            nn.LeakyReLU(0.2)
        )
        
        self.branch3x3 = nn.Sequential(
            nn.Conv2d(in_channels, branch3x3_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(branch3x3_ch),
            nn.LeakyReLU(0.2)
        )
        
        self.branch5x5 = nn.Sequential(
            nn.Conv2d(in_channels, branch5x5_ch, kernel_size=5, padding=2),
            nn.BatchNorm2d(branch5x5_ch),
            nn.LeakyReLU(0.2)
        )

    def forward(self, x):
        return torch.cat([self.branch1x1(x), self.branch3x3(x), self.branch5x5(x)], dim=1)

In [54]:
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super(SEBlock, self).__init__()
        self.squeeze = nn.AdaptiveAvgPool2d(1)
        self.excitation = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.squeeze(x).view(b, c)
        y = self.excitation(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

In [55]:
class DeepFakeModel_SkipConnectionBlock(nn.Module):
    def __init__(self):
        super(DeepFakeModel_SkipConnectionBlock, self).__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2)
        )

        self.stage1 = self._make_layer(32, 64)
        self.stage2 = self._make_layer(64, 128)
        self.stage3 = self._make_layer(128, 256)
        self.stage4 = self._make_layer(256, 512)

        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(512, 1),
        )


    def _make_layer(self, in_channel, out_channel):
        return nn.Sequential(
            SkipConnectionBlock(in_channel),
            self._transition(in_channel, out_channel)
        )

    def _transition(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.LeakyReLU(0.2)
        )
    
    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.stage4(x)

        x = self.gap(x)

        return self.classifier(x)


In [56]:
class DeepFakeModel_MultiScaleBlock(nn.Module):
    def __init__(self):
        super(DeepFakeModel_MultiScaleBlock, self).__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2)
        )

        self.stage1 = self._make_layer(32, 64)
        self.stage2 = self._make_layer(64, 128)
        self.stage3 = self._make_layer(128, 256)
        self.stage4 = self._make_layer(256, 512)

        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(512, 1),
        )

    def _make_layer(self, in_channel, out_channel):
        return nn.Sequential(
            MultiScaleBlock(in_channel, in_channel),
            self._transition(in_channel, out_channel)
        )

    def _transition(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.LeakyReLU(0.2)
        )
    
    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.stage4(x)

        x = self.gap(x)

        return self.classifier(x)

In [57]:
class DeepFakeModel_SEBlock(nn.Module):
    def __init__(self):
        super(DeepFakeModel_SEBlock, self).__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2),
            SEBlock(32)
        )

        self.stage1 = self._make_layer(32, 64)
        self.stage2 = self._make_layer(64, 128)
        self.stage3 = self._make_layer(128, 256)
        self.stage4 = self._make_layer(256, 512)

        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(512, 1),
        )

    def _make_layer(self, in_channel, out_channel):
        return nn.Sequential(
            MultiScaleBlock(in_channel, in_channel),
            SEBlock(in_channel),
            self._transition(in_channel, out_channel)
        )

    def _transition(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.LeakyReLU(0.2)
        )
    
    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.stage4(x)

        x = self.gap(x)
        return self.classifier(x)

In [58]:
model1 = DeepFakeModel_SkipConnectionBlock().to(device)
model2 = DeepFakeModel_MultiScaleBlock().to(device)
model3 = DeepFakeModel_SEBlock().to(device)


num_negatve = 41499
num_positive = 8500
pos_weight_value = torch.tensor([num_negatve/num_positive]).to(device)

loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight_value)

optimizer1 = torch.optim.AdamW(model1.parameters(), lr=1e-4, weight_decay=0.01)
optimizer2 = torch.optim.AdamW(model2.parameters(), lr=1e-4, weight_decay=0.01)
optimizer3 = torch.optim.AdamW(model3.parameters(), lr=1e-4, weight_decay=0.01)

scheduler1 = lr_scheduler.CosineAnnealingLR(optimizer=optimizer1, T_max=80, eta_min=1e-6)
scheduler2 = lr_scheduler.CosineAnnealingLR(optimizer=optimizer2, T_max=80, eta_min=1e-6)
scheduler3 = lr_scheduler.CosineAnnealingLR(optimizer=optimizer3, T_max=80, eta_min=1e-6)



In [59]:
print(summary(model1))
print(summary(model2))
print(summary(model3))

Layer (type:depth-idx)                   Param #
DeepFakeModel_SkipConnectionBlock        --
├─Sequential: 1-1                        --
│    └─Conv2d: 2-1                       2,432
│    └─BatchNorm2d: 2-2                  64
│    └─LeakyReLU: 2-3                    --
├─Sequential: 1-2                        --
│    └─SkipConnectionBlock: 2-4          --
│    │    └─Sequential: 3-1              18,624
│    └─Sequential: 2-5                   --
│    │    └─Conv2d: 3-2                  18,496
│    │    └─BatchNorm2d: 3-3             128
│    │    └─LeakyReLU: 3-4               --
├─Sequential: 1-3                        --
│    └─SkipConnectionBlock: 2-6          --
│    │    └─Sequential: 3-5              74,112
│    └─Sequential: 2-7                   --
│    │    └─Conv2d: 3-6                  73,856
│    │    └─BatchNorm2d: 3-7             256
│    │    └─LeakyReLU: 3-8               --
├─Sequential: 1-4                        --
│    └─SkipConnectionBlock: 2-8          --
│    │

In [60]:
def response(id_img, target):
    data = {
        'id': id_img,
        'target': target
    }

    df = pd.DataFrame(data)
    df.to_csv('result.csv', index=False)

In [61]:
num_epoch = 80

# Переводим всё в режим обучения
model1.train(); model2.train(); model3.train()

for epoch in range(num_epoch):
    running_loss1 = 0.0
    running_loss2 = 0.0
    running_loss3 = 0.0
    
    loop = tqdm(face_dataset_loader, desc=f"Epoch [{epoch+1}/{num_epoch}]")
    
    for X, y in loop:
        X = X.to(device)
        y = y.to(device).float().unsqueeze(1)

        optimizer1.zero_grad()
        predict1 = model1(X)
        loss1 = loss_fn(predict1, y)
        loss1.backward()
        optimizer1.step()
        
        optimizer2.zero_grad()
        predict2 = model2(X)
        loss2 = loss_fn(predict2, y)
        loss2.backward()
        optimizer2.step()
        
        optimizer3.zero_grad()
        predict3 = model3(X)
        loss3 = loss_fn(predict3, y) 
        loss3.backward()
        optimizer3.step()

        running_loss1 += loss1.item()
        running_loss2 += loss2.item()
        running_loss3 += loss3.item()
        
        loop.set_postfix(l1=loss1.item(), l2=loss2.item(), l3=loss3.item())

    scheduler1.step()
    scheduler2.step()
    scheduler3.step()

    current_lr = scheduler1.get_last_lr()[0]
    print(f"Epoch {epoch+1}: LR = {current_lr:.2e}")

torch.save(model1.state_dict(), "model1_resnet.pth")
torch.save(model2.state_dict(), "model2_multiscale.pth")
torch.save(model3.state_dict(), "model3_dense.pth")

print("Обучение завершено и модели сохранены!")

Epoch [1/80]: 100%|██████████| 782/782 [09:25<00:00,  1.38it/s, l1=0.612, l2=0.539, l3=0.599]


Epoch 1: LR = 1.00e-04


Epoch [2/80]: 100%|██████████| 782/782 [09:18<00:00,  1.40it/s, l1=0.794, l2=0.706, l3=0.887]


Epoch 2: LR = 9.98e-05


Epoch [3/80]: 100%|██████████| 782/782 [09:15<00:00,  1.41it/s, l1=0.877, l2=0.913, l3=0.864]


...

Epoch 78: LR = 1.15e-06


Epoch [79/80]: 100%|██████████| 782/782 [09:17<00:00,  1.40it/s, l1=0.00117, l2=0.0129, l3=0.00298]  


Epoch 79: LR = 1.04e-06


Epoch [80/80]: 100%|██████████| 782/782 [09:15<00:00,  1.41it/s, l1=0.00177, l2=0.17, l3=0.0308]     


Epoch 80: LR = 1.00e-06
Обучение завершено и модели сохранены!


In [65]:
id_img = []
targets = []

model1.eval()
model2.eval()
model3.eval()

with torch.no_grad():
    for X, names in face_dataset_validation_loader:
        X = X.to(device)
        
        out1 = model1(X)
        out2 = model2(X)
        out3 = model3(X)
        
        prob1 = torch.sigmoid(out1.squeeze())
        prob2 = torch.sigmoid(out2.squeeze())
        prob3 = torch.sigmoid(out3.squeeze())
        
        ensemble_prob = (prob1 + prob2 + prob3) / 3.0

        predict = (ensemble_prob > 0.5).int()
        
        id_img.extend(names)
        
        if predict.dim() == 0: 
            targets.append(predict.item())
        else:
            targets.extend(predict.cpu().tolist())

response(id_img, targets)